# Modelos supervisionados e validação

Percurso dos capítulos 7–13 e avaliação final dos exemplos Wisconsin e pinguins. As comparações e decisões são calculadas no desenvolvimento. O teste só é aberto na secção final, depois de guardar as escolhas.

Execute as células pela ordem apresentada. No Colab, use o ZIP completo dos materiais de apoio; no computador, mantenha este caderno na pasta principal. Guarde as suas interpretações em células de texto. As saídas são calculadas quando executar o caderno.


## Preparar os ficheiros


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import zipfile

# No computador, abra este caderno junto de reproduzir.py.
# No Colab, a primeira execução pede o ZIP completo dos materiais.
candidatos = [Path.cwd(), Path.cwd().parent]
pasta_colab = Path('/content/materiais_biociencias')
if pasta_colab.exists():
    candidatos += [p.parent for p in pasta_colab.rglob('reproduzir.py')]
raiz = next((p for p in candidatos if (p / 'reproduzir.py').is_file()), None)
if raiz is None:
    try:
        from google.colab import files
    except ImportError as erro:
        raise FileNotFoundError(
            'Abra o caderno na pasta que contém reproduzir.py, dados e reproducao.'
        ) from erro
    carregados = files.upload()
    arquivos = [Path(nome) for nome in carregados if nome.lower().endswith('.zip')]
    if len(arquivos) != 1:
        raise ValueError('Carregue um único ZIP completo dos materiais de apoio.')
    pasta_colab.mkdir(parents=True, exist_ok=True)
    destino = pasta_colab.resolve()
    with zipfile.ZipFile(arquivos[0]) as arquivo:
        for item in arquivo.infolist():
            caminho = (destino / item.filename).resolve()
            if not caminho.is_relative_to(destino):
                raise ValueError('O ZIP contém um caminho fora da pasta de destino.')
        arquivo.extractall(destino)
    encontrados = list(destino.rglob('reproduzir.py'))
    if len(encontrados) != 1:
        raise ValueError('O ZIP deve conter uma única pasta completa dos materiais.')
    raiz = encontrados[0].parent
os.chdir(raiz)
print('Pasta de trabalho:', raiz.name)

def executar_python(script, *argumentos):
    subprocess.run([sys.executable, script, *argumentos], check=True, cwd=raiz)


## Conferir o ambiente

Esta célula instala as dependências apenas se necessário. Se pedir um reinício, reinicie o ambiente e volte à primeira célula.


In [ ]:
from importlib.metadata import version, PackageNotFoundError

requisitos = {}
for linha in Path('requisitos.txt').read_text().splitlines():
    if '==' in linha and not linha.lstrip().startswith('#'):
        nome, esperada = linha.strip().split('==', 1)
        requisitos[nome] = esperada
alterar = False
for nome, esperada in requisitos.items():
    try:
        atual = version(nome)
    except PackageNotFoundError:
        atual = 'não instalada'
    print(f'{nome}: {atual} (referência: {esperada})')
    alterar = alterar or atual != esperada
if alterar:
    ja_importadas = any(n in sys.modules for n in ['numpy', 'pandas', 'sklearn', 'matplotlib', 'scipy'])
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', 'requisitos.txt'], check=True)
    if ja_importadas:
        raise RuntimeError(
            'Dependências instaladas. Reinicie o ambiente e execute desde a primeira célula '
            'para usar as versões agora instaladas.'
        )
else:
    print('O ambiente já tem as versões de referência.')


## 1. Ler o protocolo

Confirme as entradas, a classe positiva, as divisões e os critérios.


In [ ]:
import json
import pandas as pd
from IPython.display import display, Image
protocolo = json.loads(Path('reproducao/modelos/protocolo.json').read_text())
print(json.dumps(protocolo, ensure_ascii=False, indent=2))


## 2. Executar o desenvolvimento

São comparadas as configurações indicadas no livro. O ficheiro de decisão é guardado sem abrir as respostas do teste.


In [ ]:
executar_python('reproducao/modelos/reproduzir_modelos.py', 'desenvolvimento')


In [ ]:
pasta = Path('reproducao/modelos/resultados')
display(pd.read_csv(pasta / 'wisconsin_validacao_resumo.csv'))
display(pd.read_csv(pasta / 'pinguins_validacao_resumo.csv'))
print((pasta / 'decisao.json').read_text())


## 3. Inspecionar previsões e diagnósticos

Os rótulos das saídas distinguem medidas reunidas e médias por parte. As curvas por época usam a divisão interna própria descrita no protocolo.


In [ ]:
figuras = Path('reproducao/modelos/figuras')
for nome in ['u8b_roc.png', 'u11_curvas8.png', 'u13_calibracao.png', 'u13_permutacao.png']:
    display(Image(filename=str(figuras / nome)))


## 4. Reproduzir a avaliação final

Esta etapa lê a decisão já guardada e abre os dados de teste. Execute-a depois de concluir o desenvolvimento. Como os resultados de referência são públicos, este percurso reproduz uma avaliação conhecida; não fornece um novo teste independente para configurações escolhidas depois dessa consulta.


In [ ]:
executar_python('reproducao/modelos/reproduzir_modelos.py', 'teste')


In [ ]:
display(pd.read_csv(pasta / 'wisconsin_teste_resumo.csv'))
display(pd.read_csv(pasta / 'pinguins_teste_resumo.csv'))
display(Image(filename=str(figuras / 'u14_matriz.png')))
display(Image(filename=str(figuras / 'u14_residuos.png')))


## Concluir

Relacione a medida de desempenho com a pergunta. Compare com a referência, identifique os erros relevantes e explicite uma limitação. Os resultados Wisconsin não estabelecem um procedimento de utilização clínica.
